# NB178 — The Geometric Gravity Dictionary

**Goal**: Translate the gravity hierarchy M_Pl/M_Z = 240⁴ × 7⁹ into the concentric
sphere arena S² × R⁺ established in NB177. Every piece of the formula acquires a
geometric reading in terms of Gaussian curvatures, sphere radii, and area ratios.

**Established**: NB177 (#303–#308) gave us primorial radii r_k = P_k, area ratios
= Γ̃ eigenvalues = p_k², Gauss-Bonnet K_k×A_k = 4π, and geometric Laplacian
Γ_geom with diagonal 2K_k. NB121–123 (#261–#268) established the gravity hierarchy,
bilateral-chirality crossing, and triple convergence of 240.

**New identities**:
- #309: Path graph reduction — D Γ_geom D = tridiag(2,−1)
- #310: Metric radii sum — (D g_R⁻¹ D)[k,k] = r_k + r_{k+1}
- #311: Gauge-gravity as radii sum — r₃ + r₄ = Tr(L) = 240
- #312: Twin prime bracket — {p₃, p₄} = {P₂−1, P₂+1} twin primes around P₂
- #313: Gravity in curvature language — M_Pl/M_Z in pure K_k

In [2]:
# ── Setup ──
import sys, numpy as np
from pathlib import Path
from fractions import Fraction
from functools import reduce
import operator

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, RHO

primes = list(SA.primes)   # [2, 3, 5, 7]
P = [1]                     # primorial radii P_k
for p in primes:
    P.append(P[-1] * p)
# P = [1, 2, 6, 30, 210]

phi_P4 = len(SA.Z_star)  # 48

print("Primorial radii r_k = P_k:", P)
print(f"Primes: {primes}")
print(f"P₄ = {P[4]}, φ(P₄) = {phi_P4}")
print(f"ρ = 1/√P₄ = {float(RHO):.6f}")

Primorial radii r_k = P_k: [1, 2, 6, 30, 210]
Primes: [2, 3, 5, 7]
P₄ = 210, φ(P₄) = 48
ρ = 1/√P₄ = 0.069007


## Established: The Concentric Sphere Arena (NB177)

NB177 established that the S² × R⁺ arena assigns **primorial radii** $r_k = P_k$ to each
covering level, with area ratios $A_{k+1}/A_k = p_{k+1}^2$ equal to the Γ̃ eigenvalues,
Gaussian curvature $K_k = 1/P_k^2$, and a geometric Laplacian $\Gamma_\text{geom}$ with
diagonal entries $2K_k$.

NB121–123 established the gravity hierarchy $M_\text{Pl}/M_Z = 240^4 \times 7^9$ where
$240 = \text{Tr}(L) = P_3 \times \varphi(P_3)$ and the exponent $9 = \sigma_3(p_1) = d_0^2$
is the bilateral-chirality crossing (#263).

**Goal**: Give every piece of $240^4 \times 7^9$ a geometric meaning in terms of
curvatures, radii, and areas of the concentric spheres $S^2(P_k)$.

In [3]:
# ── Build the two operators: Γ_geom (NB177) and g_R^{-1} (NB82) ──

# Covering Jacobian J: 4×5, R_k = p_{k+1}·θ_{k+1} - θ_k
J = np.zeros((4, 5))
for k in range(4):
    J[k, k] = -1
    J[k, k+1] = primes[k]

# Sphere metric: G = diag(P_k²) = diag(areas/4π)
G_inv = np.diag([Fraction(1, P[k]**2) for k in range(5)])

# Geometric Laplacian: Γ_geom = J G⁻¹ Jᵀ
Gamma_exact = [[Fraction(0)]*4 for _ in range(4)]
for i in range(4):
    for j in range(4):
        for m in range(5):
            Gamma_exact[i][j] += Fraction(int(J[i,m])) * Fraction(1, P[m]**2) * Fraction(int(J[j,m]))

Gamma = np.array([[float(Gamma_exact[i][j]) for j in range(4)] for i in range(4)])

# Solenoid metric inverse g_R^{-1}: tridiagonal (NB82)
g_inv = np.zeros((4, 4))
for k in range(4):
    g_inv[k, k] = (1 + primes[k]) / P[k]
    if k > 0:
        g_inv[k, k-1] = -1.0 / P[k-1]
        g_inv[k-1, k] = -1.0 / P[k-1]

# Radii normalization: D = diag(P_0, P_1, P_2, P_3)
D = np.diag([float(P[k]) for k in range(4)])

print("Γ_geom (exact diagonal: 2K_k = 2/P_k²):")
for k in range(4):
    print(f"  [{k},{k}] = {Gamma_exact[k][k]} = 2/{P[k]}²")

print(f"\ng_R⁻¹ (exact diagonal: (1+p_{{k+1}})/P_k):")
for k in range(4):
    d = Fraction(1 + primes[k], P[k])
    print(f"  [{k},{k}] = {d}")

print(f"\nD = diag({[P[k] for k in range(4)]}) = diag(inner sphere radii)")

Γ_geom (exact diagonal: 2K_k = 2/P_k²):
  [0,0] = 2 = 2/1²
  [1,1] = 1/2 = 2/2²
  [2,2] = 1/18 = 2/6²
  [3,3] = 1/450 = 2/30²

g_R⁻¹ (exact diagonal: (1+p_{k+1})/P_k):
  [0,0] = 3
  [1,1] = 2
  [2,2] = 1
  [3,3] = 4/15

D = diag([1, 2, 6, 30]) = diag(inner sphere radii)


## Identity #309 — Path Graph Reduction

**Claim**: The geometric Laplacian $\Gamma_\text{geom}$, when normalized by sphere radii
$D = \text{diag}(P_0, P_1, P_2, P_3)$, reduces to the **path graph Laplacian**:

$$D \, \Gamma_\text{geom} \, D = \begin{pmatrix} 2 & -1 & & \\ -1 & 2 & -1 & \\ & -1 & 2 & -1 \\ & & -1 & 2 \end{pmatrix} = L_\text{path}$$

This is the simplest possible discrete Laplacian on a 1D chain of 4 nodes.
Determinant $= 5 = p_3$ (the charge prime). Eigenvalues $= 2(1 - \cos(k\pi/5))$
for $k = 1, 2, 3, 4$.

In [4]:
# ── #309: Path graph reduction ──
DGD = D @ Gamma @ D

# Reference: 4×4 path Laplacian with Dirichlet boundary
L_path = np.zeros((4, 4))
for k in range(4):
    L_path[k, k] = 2
    if k > 0:
        L_path[k, k-1] = -1
        L_path[k-1, k] = -1

print("D · Γ_geom · D =")
for i in range(4):
    print(f"  [{' '.join(f'{DGD[i,j]:>6.2f}' for j in range(4))} ]")

match = np.allclose(DGD, L_path)
det_DGD = np.linalg.det(DGD)
eigs = np.sort(np.linalg.eigvalsh(DGD))

print(f"\n  = L_path (path graph Laplacian)? {match}")
print(f"  det(D Γ_geom D) = {det_DGD:.1f} = p₃ = {primes[2]}")
print(f"\n  Eigenvalues: {[f'{e:.6f}' for e in eigs]}")
print(f"  = 2(1−cos(kπ/5)) for k=1..4:")
for k in range(1, 5):
    expected = 2 * (1 - np.cos(k * np.pi / 5))
    print(f"    k={k}: {expected:.6f}")

assert match, "Path graph reduction FAILED"
assert abs(det_DGD - 5.0) < 1e-10, "Determinant ≠ p₃"
print("\n#309 PASS ✓")

D · Γ_geom · D =
  [  2.00  -1.00   0.00   0.00 ]
  [ -1.00   2.00  -1.00   0.00 ]
  [  0.00  -1.00   2.00  -1.00 ]
  [  0.00   0.00  -1.00   2.00 ]

  = L_path (path graph Laplacian)? True
  det(D Γ_geom D) = 5.0 = p₃ = 5

  Eigenvalues: ['0.381966', '1.381966', '2.618034', '3.618034']
  = 2(1−cos(kπ/5)) for k=1..4:
    k=1: 0.381966
    k=2: 1.381966
    k=3: 2.618034
    k=4: 3.618034

#309 PASS ✓


## Identity #310 — Metric Radii Sum

**Claim**: The solenoid metric inverse $g_R^{-1}$, when normalized by the same
sphere radii $D$, has diagonal entries equal to **sums of adjacent primorial radii**:

$$(D \, g_R^{-1} \, D)_{kk} = r_k + r_{k+1} = P_k + P_{k+1}$$

and off-diagonals $-r_{k+1} = -P_{k+1}$. The diagonal reads:

| $k$ | $r_k + r_{k+1}$ | Value | Structural reading |
|-----|-----------------|-------|--------------------|
| 0 | $P_0 + P_1$ | **3** | $= p_2$ (chirality prime) |
| 1 | $P_1 + P_2$ | **8** | $= \varphi(P_3)$ (Euler totient) |
| 2 | $P_2 + P_3$ | **36** | $= P_2^2$ (charge sphere radius²) |
| 3 | $P_3 + P_4$ | **240** | $= \text{Tr}(L)$ (gauge-gravity bridge) |

Each entry has a distinct structural identity.

In [5]:
# ── #310: Metric radii sum ──
DgD = D @ g_inv @ D

print("D · g_R⁻¹ · D =")
for i in range(4):
    print(f"  [{' '.join(f'{DgD[i,j]:>8.1f}' for j in range(4))} ]")

print("\nDiagonal verification (expect r_k + r_{k+1}):")
labels = ["p₂", "φ(P₃)", "P₂²", "Tr(L)"]
for k in range(4):
    expected = P[k] + P[k+1]
    actual = DgD[k, k]
    print(f"  k={k}: {actual:.0f} = P_{k}+P_{k+1} = {P[k]}+{P[k+1]} = {expected}  [{labels[k]}]")
    assert abs(actual - expected) < 1e-10, f"Diagonal [{k}] mismatch"

print("\nOff-diagonal verification (expect -r_{k+1}):")
for k in range(3):
    expected = -P[k+1]
    actual = DgD[k, k+1]
    print(f"  [{k},{k+1}]: {actual:.0f} = -P_{k+1} = {expected}")
    assert abs(actual - expected) < 1e-10, f"Off-diagonal [{k},{k+1}] mismatch"

# The sub-identities within the diagonal:
print("\nSub-identities:")
print(f"  r₀+r₁ = {P[0]+P[1]} = p₂ = {primes[1]}? {P[0]+P[1] == primes[1]}")
phi_P3 = reduce(operator.mul, [p-1 for p in primes[:3]])
print(f"  r₁+r₂ = {P[1]+P[2]} = φ(P₃) = {phi_P3}? {P[1]+P[2] == phi_P3}")
print(f"  r₂+r₃ = {P[2]+P[3]} = P₂² = {P[2]**2}? {P[2]+P[3] == P[2]**2}")
print(f"  r₃+r₄ = {P[3]+P[4]} = Tr(L) = 240? {P[3]+P[4] == 240}")

print("\n#310 PASS ✓")

D · g_R⁻¹ · D =
  [     3.0     -2.0      0.0      0.0 ]
  [    -2.0      8.0     -6.0      0.0 ]
  [     0.0     -6.0     36.0    -30.0 ]
  [     0.0      0.0    -30.0    240.0 ]

Diagonal verification (expect r_k + r_{k+1}):
  k=0: 3 = P_0+P_1 = 1+2 = 3  [p₂]
  k=1: 8 = P_1+P_2 = 2+6 = 8  [φ(P₃)]
  k=2: 36 = P_2+P_3 = 6+30 = 36  [P₂²]
  k=3: 240 = P_3+P_4 = 30+210 = 240  [Tr(L)]

Off-diagonal verification (expect -r_{k+1}):
  [0,1]: -2 = -P_1 = -2
  [1,2]: -6 = -P_2 = -6
  [2,3]: -30 = -P_3 = -30

Sub-identities:
  r₀+r₁ = 3 = p₂ = 3? True
  r₁+r₂ = 8 = φ(P₃) = 8? True
  r₂+r₃ = 36 = P₂² = 36? True
  r₃+r₄ = 240 = Tr(L) = 240? True

#310 PASS ✓


## Identity #311 — Gauge-Gravity as Radii Sum

**Claim**: The gauge-gravity bridge number 240 = Tr(L) is the **sum of the two
outermost primorial radii**:

$$r_3 + r_4 = P_3 + P_4 = 30 + 210 = 240 = \text{Tr}(L)$$

This identity uses the four-prime equation $p_4 + 1 = \varphi(P_3)$, i.e.
$7 + 1 = (2-1)(3-1)(5-1) = 8$, so that $P_3 + P_4 = P_3(1 + p_4) = P_3 \cdot \varphi(P_3)$.

**Uniqueness**: The combined constraint $p_4 - 1 = P_2$ (#267) AND $p_4 + 1 = \varphi(P_3)$
is satisfied **only** by $\{2, 3, 5, 7\}$ among all prime quadruples.

In [6]:
# ── #311: Gauge-gravity as radii sum ──
from sympy import isprime as is_prime

r3_plus_r4 = P[3] + P[4]
Tr_L = 240

print(f"r₃ + r₄ = P₃ + P₄ = {P[3]} + {P[4]} = {r3_plus_r4}")
print(f"Tr(L)    = P₃ × φ(P₃) = {P[3]} × {phi_P3} = {Tr_L}")
print(f"Equal? {r3_plus_r4 == Tr_L}")
print()
print(f"WHY: P₃(1+p₄) = {P[3]}×(1+{primes[3]}) = {P[3]*(1+primes[3])}")
print(f"     P₃×φ(P₃) = {P[3]}×{phi_P3} = {P[3]*phi_P3}")
print(f"     Equal because 1+p₄ = φ(P₃): 1+{primes[3]} = {phi_P3}")

# Uniqueness: combined p4-1=P2 AND p4+1=phi(P3)
print("\nUniqueness check (p₄-1=P₂ AND p₄+1=φ(P₃)):")
all_primes = [p for p in range(2, 100) if is_prime(p)]
hits = []
for i in range(len(all_primes)):
    for j in range(i+1, len(all_primes)):
        for k in range(j+1, len(all_primes)):
            for m in range(k+1, len(all_primes)):
                ps = [all_primes[i], all_primes[j], all_primes[k], all_primes[m]]
                p2 = ps[0]*ps[1]
                phi_p3 = reduce(operator.mul, [p-1 for p in ps[:3]])
                if ps[3]-1 == p2 and ps[3]+1 == phi_p3:
                    hits.append(ps)
                    print(f"  {ps}: p₄-1={ps[3]-1}=P₂={p2} ✓, p₄+1={ps[3]+1}=φ(P₃)={phi_p3} ✓")

assert len(hits) == 1 and hits[0] == [2,3,5,7], "Uniqueness FAILED"
print(f"  → UNIQUE: only {{2,3,5,7}} satisfies both constraints (searched p < 100)")

assert r3_plus_r4 == Tr_L
print("\n#311 PASS ✓")

r₃ + r₄ = P₃ + P₄ = 30 + 210 = 240
Tr(L)    = P₃ × φ(P₃) = 30 × 8 = 240
Equal? True

WHY: P₃(1+p₄) = 30×(1+7) = 240
     P₃×φ(P₃) = 30×8 = 240
     Equal because 1+p₄ = φ(P₃): 1+7 = 8

Uniqueness check (p₄-1=P₂ AND p₄+1=φ(P₃)):
  [2, 3, 5, 7]: p₄-1=6=P₂=6 ✓, p₄+1=8=φ(P₃)=8 ✓
  → UNIQUE: only {2,3,5,7} satisfies both constraints (searched p < 100)

#311 PASS ✓


## Identity #312 — The Consecutive Quartet

**Claim**: The four numbers $p_3, P_2, p_4, \varphi(P_3)$ form **four consecutive
integers**:

$$5, \; 6, \; 7, \; 8 \;=\; p_3, \;\; p_1 p_2, \;\; p_4, \;\; \prod_{k=1}^{3}(p_k - 1)$$

This means:
- $p_3 + 1 = P_2 = p_1 p_2$ (charge prime + 1 = bilateral × chirality)
- $p_4 - 1 = P_2$ (ultimation prime − 1 = first sub-primorial, #267)
- $p_4 + 1 = \varphi(P_3)$ (ultimation prime + 1 = Euler totient of third primorial)
- $\{p_3, p_4\} = \{P_2 - 1, P_2 + 1\}$ are twin primes bracketing $P_2$

The resulting chain $P_2 - 1, P_2, P_2 + 1, P_2 + 2 = p_3, P_2, p_4, \varphi(P_3)$
is **unique** to $\{2, 3, 5, 7\}$ among all prime quadruples.

In [7]:
# ── #312: The consecutive quartet ──
p3, P2, p4 = primes[2], P[2], primes[3]
phi_P3_val = reduce(operator.mul, [p-1 for p in primes[:3]])

quartet = [p3, P2, p4, phi_P3_val]
print(f"p₃ = {p3}, P₂ = {P2}, p₄ = {p4}, φ(P₃) = {phi_P3_val}")
print(f"Quartet: {quartet}")
print(f"Consecutive? {quartet == list(range(quartet[0], quartet[0]+4))}")
print()

# Individual conditions
print("Component conditions:")
print(f"  p₃ + 1 = P₂?   {p3}+1 = {p3+1} = P₂ = {P2}:  {p3+1 == P2}")
print(f"  p₄ - 1 = P₂?   {p4}-1 = {p4-1} = P₂ = {P2}:  {p4-1 == P2}")
print(f"  p₄ + 1 = φ(P₃)? {p4}+1 = {p4+1} = φ(P₃) = {phi_P3_val}: {p4+1 == phi_P3_val}")
print(f"  Twin primes?     {p3} and {p4} differ by {p4-p3}: {p4-p3 == 2}")

# Uniqueness: all four conditions simultaneously
print("\nUniqueness (all four consecutive):")
count = 0
for i in range(len(all_primes)):
    for j in range(i+1, len(all_primes)):
        for k in range(j+1, len(all_primes)):
            for m in range(k+1, len(all_primes)):
                ps = [all_primes[i], all_primes[j], all_primes[k], all_primes[m]]
                pp2 = ps[0]*ps[1]
                pphi = reduce(operator.mul, [p-1 for p in ps[:3]])
                q = [ps[2], pp2, ps[3], pphi]
                if q == list(range(q[0], q[0]+4)):
                    count += 1
                    print(f"  {ps} → {q} (consecutive ✓)")

assert count == 1, f"Expected unique, found {count}"
print(f"  → UNIQUE among all prime quadruples with p < 100")

assert quartet == list(range(5, 9))
print("\n#312 PASS ✓")

p₃ = 5, P₂ = 6, p₄ = 7, φ(P₃) = 8
Quartet: [5, 6, 7, 8]
Consecutive? True

Component conditions:
  p₃ + 1 = P₂?   5+1 = 6 = P₂ = 6:  True
  p₄ - 1 = P₂?   7-1 = 6 = P₂ = 6:  True
  p₄ + 1 = φ(P₃)? 7+1 = 8 = φ(P₃) = 8: True
  Twin primes?     5 and 7 differ by 2: True

Uniqueness (all four consecutive):
  [2, 3, 5, 7] → [5, 6, 7, 8] (consecutive ✓)
  → UNIQUE among all prime quadruples with p < 100

#312 PASS ✓


## Identity #313 — Gravity in Curvature Language

**Claim**: The entire gravity hierarchy $M_\text{Pl}/M_Z = 240^4 \times 7^9$ can be
written in terms of Gaussian curvatures $K_k = 1/P_k^2$ of the concentric spheres:

$$\frac{M_\text{Pl}}{M_Z} = \left[\prod_{k=0}^{2} \sqrt{\frac{K_k}{K_{k+1}}} \left(\sqrt{\frac{K_k}{K_{k+1}}} - 1\right)\right]^4 \times \left(\sqrt{\frac{K_3}{K_4}}\right)^{K_1/K_2}$$

where:
- $\sqrt{K_k/K_{k+1}} = p_{k+1}$ (winding number = curvature root ratio)
- The product gives $\prod p_k(p_k-1) = 240$ over the inner 3 transitions
- The exponent 4 = $\omega(P_4)$ = number of covering transitions
- The base $\sqrt{K_3/K_4} = p_4 = 7$ (outermost winding)
- The ultimation exponent $K_1/K_2 = p_2^2 = 9 = \sigma_3(p_1)$ (area ratio at chirality)

This is a **geometric dictionary**, not a derivation. The algebraic identity (#261–#268)
precedes the geometric reading.

In [8]:
# ── #313: Gravity in curvature language ──
import math

# Gaussian curvatures K_k = 1/P_k²
K = [Fraction(1, P[k]**2) for k in range(5)]

print("Gaussian curvatures K_k = 1/r_k²:")
for k in range(5):
    print(f"  K_{k} = 1/{P[k]}² = {K[k]} = {float(K[k]):.2e}")

# Curvature ratios at each transition
print("\nCurvature ratios (K_k/K_{k+1} = p_{k+1}²):")
gauge_product = Fraction(1)
for k in range(4):
    ratio = K[k] / K[k+1]
    root = primes[k]   # √(K_k/K_{k+1}) = p_{k+1}
    excess = root * (root - 1)   # p(p-1)
    print(f"  K_{k}/K_{k+1} = {ratio} = {primes[k]}² = {primes[k]**2},"
          f"  √ = {root}, excess = {root}×{root-1} = {excess}")
    if k < 3:  # inner 3 only for gauge product
        gauge_product *= excess

print(f"\nGauge product (inner 3): ∏ p_k(p_k-1) = {gauge_product} = Tr(L)")
assert gauge_product == 240

# Full gravity formula
omega_P4 = 4    # number of transitions
sigma3_p1 = 9   # K_1/K_2 = p_2² = 9
p4 = primes[3]  # outermost winding = 7

M_ratio_formula = Fraction(gauge_product)**omega_P4 * Fraction(p4)**sigma3_p1
M_ratio_direct = Fraction(240)**4 * Fraction(7)**9

print(f"\nGravity formula:")
print(f"  [∏ √(K_k/K_{{k+1}})(√(K_k/K_{{k+1}})-1)]^4 × √(K₃/K₄)^{{K₁/K₂}}")
print(f"  = {gauge_product}^{omega_P4} × {p4}^{sigma3_p1}")
print(f"  = {float(M_ratio_formula):.6e}")
print(f"  = 240⁴ × 7⁹ = {float(M_ratio_direct):.6e}")
print(f"  Equal? {M_ratio_formula == M_ratio_direct}")

# Verify curvature components
print(f"\nCurvature dictionary:")
print(f"  240 = ∏ₖ₌₀² √(K_k/K_{{k+1}})(√(K_k/K_{{k+1}})-1)")
print(f"      = r₃+r₄  (radii sum, #311)")
print(f"  4   = ω(P₄)  = number of sphere transitions")
print(f"  7   = √(K₃/K₄) = outermost curvature root")
print(f"  9   = K₁/K₂  = {K[1]}/{K[2]} = {K[1]/K[2]}")
print(f"      = chirality/charge curvature ratio = σ₃(p₁)")

assert M_ratio_formula == M_ratio_direct
assert K[1]/K[2] == sigma3_p1
print("\n#313 PASS ✓")

Gaussian curvatures K_k = 1/r_k²:
  K_0 = 1/1² = 1 = 1.00e+00
  K_1 = 1/2² = 1/4 = 2.50e-01
  K_2 = 1/6² = 1/36 = 2.78e-02
  K_3 = 1/30² = 1/900 = 1.11e-03
  K_4 = 1/210² = 1/44100 = 2.27e-05

Curvature ratios (K_k/K_{k+1} = p_{k+1}²):
  K_0/K_1 = 4 = 2² = 4,  √ = 2, excess = 2×1 = 2
  K_1/K_2 = 9 = 3² = 9,  √ = 3, excess = 3×2 = 6
  K_2/K_3 = 25 = 5² = 25,  √ = 5, excess = 5×4 = 20
  K_3/K_4 = 49 = 7² = 49,  √ = 7, excess = 7×6 = 42

Gauge product (inner 3): ∏ p_k(p_k-1) = 240 = Tr(L)

Gravity formula:
  [∏ √(K_k/K_{k+1})(√(K_k/K_{k+1})-1)]^4 × √(K₃/K₄)^{K₁/K₂}
  = 240^4 × 7^9
  = 1.338836e+17
  = 240⁴ × 7⁹ = 1.338836e+17
  Equal? True

Curvature dictionary:
  240 = ∏ₖ₌₀² √(K_k/K_{k+1})(√(K_k/K_{k+1})-1)
      = r₃+r₄  (radii sum, #311)
  4   = ω(P₄)  = number of sphere transitions
  7   = √(K₃/K₄) = outermost curvature root
  9   = K₁/K₂  = 1/4/1/36 = 9
      = chirality/charge curvature ratio = σ₃(p₁)

#313 PASS ✓


## The Complete Geometric Dictionary

Every piece of the gravity hierarchy $M_\text{Pl}/M_Z = 240^4 \times 7^9$ now has
a geometric reading in the concentric sphere arena $S^2(P_k)$:

| Algebraic | Value | Geometric reading |
|-----------|-------|-------------------|
| $\text{Tr}(L)$ | 240 | $r_3 + r_4$ = sum of outermost radii (#311) |
| | | $= \prod_{k=0}^{2} \sqrt{K_k/K_{k+1}}(\sqrt{K_k/K_{k+1}}-1)$ (#313) |
| | | $= (D\,g_R^{-1}\,D)_{33}$ = last metric diagonal entry (#310) |
| $\omega(P_4)$ | 4 | number of covering transitions = dim($R$-space) |
| $p_4$ | 7 | $= r_4/r_3 = \sqrt{K_3/K_4}$ = outermost winding |
| $\sigma_3(p_1)$ | 9 | $= K_1/K_2$ = curvature ratio at chirality level |
| | | $= d_0^2$ = bilateral metric diagonal squared (#263) |

The geometric Laplacian $\Gamma_\text{geom}$ and solenoid metric $g_R^{-1}$, when
normalized by sphere radii $D = \text{diag}(P_k)$, produce:
- **Path Laplacian** $L_\text{path}$ with $\det = p_3 = 5$ (#309)
- **Radii sum matrix** with diagonal $[p_2, \varphi(P_3), P_2^2, \text{Tr}(L)]$ (#310)

The consecutive quartet $p_3, P_2, p_4, \varphi(P_3) = 5,6,7,8$ (#312) is the
number-theoretic condition that makes $r_3+r_4 = \text{Tr}(L)$ work.

**Status**: These are geometric INTERPRETATIONS of existing algebraic identities
(#261–#268). The geometry explains WHY the formula works in the $S^2 \times \mathbb{R}^+$
arena. It does NOT derive gravity from first principles — the gap "why does $M_\text{Pl}$
involve Tr($L$)?" remains open.

In [9]:
# ── Scorecard ──
print("NB178 SCORECARD — THE GEOMETRIC GRAVITY DICTIONARY")
print("=" * 65)
print()
print("# | Identity                          | Status")
print("-" * 65)
print("309 | Path graph reduction:             | STRUCTURAL, PASS")
print("    | D·Γ_geom·D = tridiag(2,-1)       | det = p₃ = 5")
print("310 | Metric radii sum:                 | STRUCTURAL, PASS")
print("    | (D·g⁻¹·D)_kk = r_k+r_{k+1}      | [3,8,36,240]")
print("311 | Gauge-gravity as radii sum:       | STRUCTURAL, PASS")
print("    | r₃+r₄ = Tr(L) = 240              | UNIQUE to {2,3,5,7}")
print("312 | Consecutive quartet:              | STRUCTURAL, PASS")
print("    | p₃,P₂,p₄,φ(P₃) = 5,6,7,8        | UNIQUE to {2,3,5,7}")
print("313 | Gravity in curvature language:    | STRUCTURAL, PASS")
print("    | M_Pl/M_Z in pure K_k of S²(P_k)  | Dictionary, not derivation")
print("-" * 65)
print()
print(f"Running total: 313 structural identities, 0 free parameters")
print(f"  (5 new in this notebook, geometric restatements of #261-#268)")
print(f"  Gravity derivation gap remains OPEN.")

NB178 SCORECARD — THE GEOMETRIC GRAVITY DICTIONARY

# | Identity                          | Status
-----------------------------------------------------------------
309 | Path graph reduction:             | STRUCTURAL, PASS
    | D·Γ_geom·D = tridiag(2,-1)       | det = p₃ = 5
310 | Metric radii sum:                 | STRUCTURAL, PASS
    | (D·g⁻¹·D)_kk = r_k+r_{k+1}      | [3,8,36,240]
311 | Gauge-gravity as radii sum:       | STRUCTURAL, PASS
    | r₃+r₄ = Tr(L) = 240              | UNIQUE to {2,3,5,7}
312 | Consecutive quartet:              | STRUCTURAL, PASS
    | p₃,P₂,p₄,φ(P₃) = 5,6,7,8        | UNIQUE to {2,3,5,7}
313 | Gravity in curvature language:    | STRUCTURAL, PASS
    | M_Pl/M_Z in pure K_k of S²(P_k)  | Dictionary, not derivation
-----------------------------------------------------------------

Running total: 313 structural identities, 0 free parameters
  (5 new in this notebook, geometric restatements of #261-#268)
  Gravity derivation gap remains OPEN.
